In [ ]:
#==================================
# 地形指标提取脚本 - 获取 Mean_Slope Northness Eastness Solar_Radiation_Index Terrain_Heterogeneity
# 作者: King 
# 日期: 2026-02-19
#==================================

import arcpy
import os
import math
import csv
import time

# ==================== 参数配置 ====================
# 1. 原始完整 DEM 文件路径 (已修改：从文件夹路径变为具体的 DEM 文件或图层路径)
dem_file = r"C:\Users\King\Documents\ArcGIS\Projects\MyProject8\MyProject8.gdb\Extract_buff1"  #数据集约300GB 1m*1m DEM 高程数据 已预先裁剪
# 2. 结果导出路径
output_csv = r"C:\Users\King\Desktop\大三下学习生活\prj-Alban\data\output\Bird_Habitat_Terrain_Stats_5km.csv"
# 3. 缓冲区图层名称
buffer_layer = r"C:\Users\King\Documents\ArcGIS\Projects\MyProject8\MyProject8.gdb\Buffer_5km"

if not os.path.exists(os.path.dirname(output_csv)):
    os.makedirs(os.path.dirname(output_csv))

arcpy.CheckOutExtension("Spatial")

# ==================== 初始化 ====================
#创建buffer图层对象
arcpy.management.MakeFeatureLayer(buffer_layer, "buffer_lyr")

oid_field = arcpy.Describe("buffer_lyr").OIDFieldName

res = arcpy.management.GetCount("buffer_lyr")
total_points = int(res[0])
results = []
header = ["OID", "Mean_Slope", "Northness", "Eastness", "Solar_Radiation_Index", "Terrain_Heterogeneity"]

start_time = time.time()
processed_count = 0
error_count = 0

print(f"--- 任务启动：共计 {total_points} 个样点准备分析 ---")
arcpy.AddMessage(f"开始地形指标提取，总数: {total_points}")

# ==================== Core ====================
# 遍历创建好的图层
with arcpy.da.SearchCursor("buffer_lyr", ["OID@"]) as cursor:
    for row in cursor:
        oid = row[0]
        
        try:
            # 修改 1: 根据 OID 选择当前的唯一缓冲区多边形
            query = f"{oid_field} = {oid}"
            arcpy.management.SelectLayerByAttribute("buffer_lyr", "NEW_SELECTION", query)
            print(f"正在处理 OID {oid}...")  # Debug 输出当前处理的 OID
            
            # 修改 2: 按刚才选中的缓冲区掩膜提取 DEM 到内存
            extracted_dem = arcpy.sa.ExtractByMask(dem_file, "buffer_lyr")
            print(f"正在处理 按掩膜提取 完毕 OID {oid}...")  # Debug 输出当前处理的 OID

            #程序运行后始终无法完成按掩膜提取 (ExtractByMask)，且没有任何报错信息
 
            # 地形计算 (将原来的 dem_path 替换为了 extracted_dem)
            slope_rast = arcpy.sa.Slope(extracted_dem, "DEGREE")
            aspect_rast = arcpy.sa.Aspect(extracted_dem)
            
            # 换算弧度并计算 Northness/Eastness
            aspect_rad = arcpy.sa.Raster(aspect_rast) * (math.pi / 180.0)
            northness_rast = arcpy.sa.Cos(aspect_rad)
            eastness_rast = arcpy.sa.Sin(aspect_rad)
            
            # (PSR)
            slope_rad = arcpy.sa.Raster(slope_rast) * (math.pi / 180.0)
            psr_rast = arcpy.sa.Cos(slope_rad) + (arcpy.sa.Sin(slope_rad) * arcpy.sa.Cos(aspect_rad))
            
            # 统计值
            m_slope = float(arcpy.management.GetRasterProperties(slope_rast, "MEAN").getOutput(0))
            std_slope = float(arcpy.management.GetRasterProperties(slope_rast, "STD").getOutput(0))
            m_north = float(arcpy.management.GetRasterProperties(northness_rast, "MEAN").getOutput(0))
            m_east = float(arcpy.management.GetRasterProperties(eastness_rast, "MEAN").getOutput(0))
            m_psr = float(arcpy.management.GetRasterProperties(psr_rast, "MEAN").getOutput(0))
            
            results.append([oid, m_slope, m_north, m_east, m_psr, std_slope])
            processed_count += 1
            
            # --- Debug 输出 ---
            if processed_count % 10 == 0 or processed_count == total_points:
                elapsed = time.time() - start_time
                speed = processed_count / elapsed # 每秒处理点数
                percent = (processed_count / total_points) * 100
                remaining = total_points - processed_count
                eta_min = (remaining / speed) / 60 if speed > 0 else 0
                
                debug_msg = (f"进度: {percent:.1f}% ({processed_count}/{total_points}) | "
                             f"速度: {speed:.2f} 点/秒 | "
                             f"预计剩余: {eta_min:.1f} 分钟"
                             f",{eta_min/60:.2f} 小时")
                print(debug_msg)
                arcpy.AddMessage(debug_msg)
                
        except Exception as e:
            error_count += 1
            err_log = f"!!! 点 OID {oid} 分析出错: {str(e)}"
            print(err_log)
            arcpy.AddWarning(err_log)

            if error_count >= 7:
                print("熔断")
                arcpy.AddError("熔断: 连续错误超过阈值，终止程序。")
                break

# 清理内存中的临时图层和选择状态
arcpy.management.SelectLayerByAttribute("buffer_lyr", "CLEAR_SELECTION")
arcpy.management.Delete("buffer_lyr")

# ==================== 写入 CSV ====================
print(f"正在保存数据至 {output_csv}...")
with open(output_csv, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(header)
    writer.writerows(results)

total_time = (time.time() - start_time) / 60
print(f"--- 任务结束 ---")
print(f"成功处理: {processed_count} | 失败记录: {error_count}")
print(f"总耗时: {total_time:.2f} 分钟")
arcpy.CheckInExtension("Spatial")



# 没啥用，速度慢在 ExtractByMask 这一布，且你换成clip Raster也不管用，也是很慢，当你的Cursor循环，每次都要进行一次ExtractByMask或者clip Raster，且数据集又很大，这个过程就会非常慢。
# 总体下来大概5分钟一个点，444分钟处理了全部的点

--- 任务启动：共计 202 个样点准备分析 ---
正在处理 OID 1...
正在处理 按掩膜提取 完毕 OID 1...
